# 04 · Sentiment: what the text says vs. what the stars say

**Author:** Mohamed Elsadany · data collected from Google Play on 28 Oct 2026

**My questions:**
1. How well does a sentiment model agree with the star ratings, in Arabic and in English?
2. How does sentiment differ between Arabic, English and Franco reviews?
3. Which reviews give 4–5 stars but describe a problem? These are **hidden pain points** that star ratings miss.

**Models I use** (pre-trained, from Hugging Face):
- Arabic: [`CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment`](https://huggingface.co/CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment),
  trained on **dialectal** Arabic, which fits Egyptian reviews better than models trained on formal Arabic.
- English: [`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest),
  trained on tweets, which are short and informal like app reviews.
- Franco-Arabic: no good ready-made model exists, so I use the star rating for those ~50 reviews.

**How I run it:** on Google Colab with a free GPU (*Runtime → Change runtime type → T4 GPU*). The model step takes a
few minutes there; on a laptop CPU it can take an hour or more. Predictions are saved every 2,000 reviews, so if Colab
disconnects I re-run the notebook and it continues where it stopped.

## 0. Setup

In [ ]:
# Where am I running? On Colab, I clone my repo to get the code in src/ and the folder layout.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MohamedElsadany56/egypt-fintech-reviews.git"

if IN_COLAB:
    ROOT = Path("/content/egypt-fintech-reviews")
    if not ROOT.exists():
        r = subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], capture_output=True, text=True)
        if r.returncode != 0:
            print("Could not clone the repo (is it private?). Creating the folders; upload src/topics.py in the next cell.")
            (ROOT / "src").mkdir(parents=True, exist_ok=True)
    (ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)
    (ROOT / "images").mkdir(exist_ok=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(ROOT / "src"))
print("Working in", ROOT)

In [ ]:
# The cleaned data is not in git (40 MB), so on Colab I upload it once.
# If the repo could not be cloned, I also upload src/topics.py here.
CLEAN = ROOT / "data" / "processed" / "reviews_clean.csv"
TOPICS_PY = ROOT / "src" / "topics.py"

if IN_COLAB and (not CLEAN.exists() or not TOPICS_PY.exists()):
    from google.colab import files
    print("Upload reviews_clean.csv" + ("" if TOPICS_PY.exists() else " and src/topics.py"))
    for name, data in files.upload().items():
        target = TOPICS_PY if name.endswith(".py") else CLEAN
        target.write_bytes(data)
        print("saved", name, "->", target.relative_to(ROOT))

print("reviews_clean.csv found:", CLEAN.exists(), "| topics.py found:", TOPICS_PY.exists())

In [ ]:
# Libraries. transformers and torch are already installed on Colab.
import re
import warnings

import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", message="obj.round has no effect")
pd.set_option("display.max_colwidth", 160)

try:
    import torch
    DEVICE = 0 if torch.cuda.is_available() else -1
except ImportError:
    DEVICE = -1
print("GPU available:", DEVICE == 0, "" if DEVICE == 0 else "-> this will be slow; on Colab switch the runtime to T4 GPU")

APPS = ["InstaPay", "Fawry", "Telda", "valU", "Vodafone Cash"]
INK, MUTED, GRID = "#1f1e1c", "#6b6a64", "#e6e5e0"

## 1. Which reviews I score

I score every review with 3+ words (`text_usable`). One-word reviews like "ممتاز" don't need a model: the star
rating already says it all. For the model I use the original text with light cleaning (links removed, stretched
letters shortened), not `content_clean`, because the models were trained on normal spelling.

In [ ]:
df = pd.read_csv(CLEAN, parse_dates=["at"])
df["content_clean"] = df["content_clean"].fillna("")
text = df[df["text_usable"]].copy()

URL = re.compile(r"https?://\S+|www\.\S+")
REPEATED = re.compile(r"(.)\1{2,}")
def model_text(t):
    t = URL.sub(" ", str(t))
    t = REPEATED.sub(r"\1\1", t)
    return re.sub(r"\s+", " ", t).strip()[:512]

text["model_text"] = text["content"].map(model_text)
text.groupby(["app", "lang_detected"]).size().unstack(fill_value=0).reindex(APPS)

## 2. Running the models

Results are saved to `data/processed/sentiment_predictions.csv` (review ID, model, label, confidence).
The function skips reviews that already have a prediction, so re-running only does the missing ones.

In [ ]:
PRED = ROOT / "data" / "processed" / "sentiment_predictions.csv"
MODELS = {
    "arabic": "CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment",
    "english": "cardiffnlp/twitter-roberta-base-sentiment-latest",
}

def load_predictions():
    if PRED.exists():
        return pd.read_csv(PRED)
    return pd.DataFrame(columns=["reviewId", "model", "model_label", "model_score"])

def run_model(lang, chunk=2000, batch_size=64):
    from transformers import pipeline
    done = set(load_predictions()["reviewId"])
    todo = text[(text["lang_detected"] == lang) & ~text["reviewId"].isin(done)]
    print(f"{lang}: {len(todo):,} reviews to score ({len(done):,} already saved)")
    if todo.empty:
        return
    clf = pipeline("text-classification", model=MODELS[lang], device=DEVICE)
    for start in range(0, len(todo), chunk):
        part = todo.iloc[start:start + chunk]
        preds = clf(part["model_text"].tolist(), batch_size=batch_size, truncation=True, max_length=128)
        out = pd.DataFrame({
            "reviewId": part["reviewId"].values,
            "model": MODELS[lang],
            "model_label": [p["label"].lower() for p in preds],
            "model_score": [round(p["score"], 4) for p in preds],
        })
        out.to_csv(PRED, mode="a", header=not PRED.exists(), index=False)
        print(f"  saved {min(start + chunk, len(todo)):,} / {len(todo):,}")

In [ ]:
run_model("arabic")
run_model("english")

## 3. Combining predictions with the reviews

For Franco reviews (no model) I use the star sentiment. `model_sentiment` is always one of
negative / neutral / positive, so I can compare it directly with `star_sentiment`
(1–2★ = negative, 3★ = neutral, 4–5★ = positive).

In [ ]:
pred = load_predictions().drop_duplicates("reviewId", keep="last")
s = text.merge(pred, on="reviewId", how="left")
s["model_sentiment"] = s["model_label"].where(s["model_label"].isin(["negative", "neutral", "positive"]))
franco = s["lang_detected"] == "franco"
s.loc[franco, "model_sentiment"] = s.loc[franco, "star_sentiment"]
print(f"{s['model_sentiment'].notna().mean():.1%} of text reviews have a sentiment")
print("Labels returned by the models:", sorted(pred["model_label"].unique()))
s.groupby("lang_detected")["model_sentiment"].value_counts(normalize=True).mul(100).round(1).unstack()

## 4. How well does the model agree with the stars?

Stars and text measure slightly different things, so I don't expect 100%. I check two numbers:
- **Exact agreement:** model sentiment equals star sentiment (3★ reviews make this hard: "neutral" is fuzzy).
- **Polarity agreement:** for clearly negative (1–2★) and clearly positive (4–5★) reviews only, does the model say
  negative / positive respectively? This is the fairest test of the model.

In [ ]:
m = s[~franco & s["model_sentiment"].notna()]
def agreement(g):
    clear = g[g["star_sentiment"] != "neutral"]
    return pd.Series({
        "reviews": len(g),
        "exact_agreement_%": (g["model_sentiment"] == g["star_sentiment"]).mean() * 100,
        "polarity_agreement_%": (clear["model_sentiment"] == clear["star_sentiment"]).mean() * 100,
    })
agree = m.groupby("lang_detected").apply(agreement, include_groups=False)
agree.loc["all"] = agreement(m)
agree.round(1)

In [ ]:
# Stars (rows) vs. model (columns), as % of each star group, for Arabic and English.
for lang in ["arabic", "english"]:
    g = m[m["lang_detected"] == lang]
    print(f"\n{lang}")
    display(pd.crosstab(g["score"], g["model_sentiment"], normalize="index").mul(100).round(1))

## 5. Does sentiment differ by language?

Share of negative text per app and language (inside each app's full-coverage period, like the other notebooks).
If English reviews are much more negative than Arabic ones for the same app, the users who write in English
(often younger, or living abroad) may have different problems.

In [ ]:
cov = s[s["in_full_coverage"]]
by_lang = (cov.assign(neg=cov["model_sentiment"].eq("negative"))
              .pivot_table(index="app", columns="lang_detected", values="neg", aggfunc="mean")
              .mul(100).round(1).reindex(APPS))
counts = cov.pivot_table(index="app", columns="lang_detected", values="reviewId", aggfunc="count").reindex(APPS)
print("% negative text  (cells with < 30 reviews are too small to trust)")
by_lang.where(counts >= 30)

## 6. Hidden pain points: 4–5★ reviews with negative text

These are users who still give a high rating but describe a problem ("good app but transfers keep failing").
Star-based dashboards count them as happy. I keep only predictions the model is confident about (score ≥ 0.8).

In [ ]:
pos = cov[cov["is_positive"] & cov["lang_detected"].isin(["arabic", "english"])]
pos = pos.assign(hidden=(pos["model_sentiment"] == "negative") & (pos["model_score"] >= 0.8))
hidden_share = pos.groupby("app")["hidden"].agg(["size", "mean"]).reindex(APPS)
hidden_share.columns = ["positive_text_reviews", "hidden_pain_%"]
hidden_share["hidden_pain_%"] = (hidden_share["hidden_pain_%"] * 100).round(1)
hidden_share

In [ ]:
# What are the hidden pain points about? Same topic rules as notebook 03.
from topics import TOPIC_LABELS, add_topics

hidden = add_topics(pos[pos["hidden"]])
topic_cols = [c for c in hidden.columns if c.startswith("topic_")]
ht = (hidden.groupby("app")[topic_cols].mean().T * 100).reindex(columns=APPS).round(1)
ht.index = [TOPIC_LABELS[c.replace("topic_", "")] for c in ht.index]
ht.loc[ht.max(axis=1).sort_values(ascending=False).index].head(8)

In [ ]:
# A few examples per app, to check the model and to quote in the README.
for app in APPS:
    g = hidden[(hidden["app"] == app) & (hidden["word_count"] >= 6)]
    print(f"\n=== {app}  ({len(g)} examples with 6+ words)")
    for _, r in g.sample(min(4, len(g)), random_state=1).iterrows():
        print(f"  • {r['score']}★ [{r['topics']}] {r['content'][:170]}")

In [ ]:
# Chart: share of 4-5★ written reviews that describe a problem.
order = hidden_share.sort_values("hidden_pain_%")
fig, ax = plt.subplots(figsize=(8, 3.2))
bars = ax.barh(order.index, order["hidden_pain_%"], color="#2a78d6", height=0.6)
for b, v, n in zip(bars, order["hidden_pain_%"], order["positive_text_reviews"]):
    ax.text(b.get_width() + 0.4, b.get_y() + b.get_height() / 2, f"{v:.0f}%  (of {n:,})",
            va="center", fontsize=8, color=INK)
ax.set_xlim(0, max(order["hidden_pain_%"].max() * 1.35, 5))
ax.xaxis.set_visible(False)
ax.spines[["top", "right", "bottom"]].set_visible(False)
ax.spines["left"].set_color("#b5b4ad")
ax.tick_params(axis="y", labelsize=9, colors=INK, length=0)
ax.set_title("Hidden pain points: 4–5★ written reviews whose text is negative", loc="left", fontsize=11, color=INK)
fig.tight_layout()
fig.savefig(ROOT / "images" / "hidden_pain_points.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. The opposite case: 1–2★ reviews with positive text

Usually one of three things: a user praising the app but angry about one issue, sarcasm, or someone who tapped the
wrong star. Reading a sample tells me which, and shows where the model gets confused.

In [ ]:
neg_pos = cov[cov["is_negative"] & (cov["model_sentiment"] == "positive") & (cov["model_score"] >= 0.8)]
print(f"{len(neg_pos):,} negative-star reviews with confident positive text "
      f"({len(neg_pos) / cov['is_negative'].sum():.1%} of negative-star text reviews)")
neg_pos.sample(min(12, len(neg_pos)), random_state=2)[["app", "score", "lang_detected", "content"]]

## 8. Saving and downloading the results

In [ ]:
keep = ["reviewId", "app", "score", "star_sentiment", "lang_detected", "model", "model_label",
        "model_sentiment", "model_score", "in_full_coverage"]
s[keep].to_csv(ROOT / "data" / "processed" / "reviews_sentiment.csv", index=False, encoding="utf-8-sig")
print("Saved data/processed/reviews_sentiment.csv")

if IN_COLAB:
    from google.colab import files
    for f in ["data/processed/sentiment_predictions.csv", "data/processed/reviews_sentiment.csv",
              "images/hidden_pain_points.png"]:
        files.download(str(ROOT / f))
    print("Also: File → Download → Download .ipynb to save this notebook with its outputs.")

In [ ]:
# Key numbers for my findings below.
print(f"Polarity agreement with stars: Arabic {agree.loc['arabic', 'polarity_agreement_%']:.0f}%, "
      f"English {agree.loc['english', 'polarity_agreement_%']:.0f}%")
print(f"Hidden pain points (4-5★ with negative text): {pos['hidden'].mean():.1%} overall")
print(hidden_share["hidden_pain_%"].to_string())

## What I found

*(I fill this in after running the notebook, using the numbers in the cell above.)*

1. **The models agree with the stars** for about __% of clearly positive or negative Arabic reviews and __% of English ones,
   so I can trust them for the analysis.
2. **Language:** English reviews are __ (more / less) negative than Arabic reviews for __; the biggest difference is __.
3. **Hidden pain points:** __% of 4–5★ written reviews describe a problem, highest for __ (__%).
   Most of them are about __.
4. **1–2★ reviews with positive text** are mostly __ (mixed reviews / sarcasm / wrong star).

**Next:** in `05_app_scorecard.ipynb` I combine everything into one scorecard per app and write my recommendations.